# Mechanistic propagation of MNIST adversarial perturbations

## 1. Imports, utilities, and config

This notebook uses the saved attack audit images as the perturbation source of truth. Intermediate traces use the common audit subset classified correctly by every selected target. Full-dataset outcome metrics use all 10,000 test images.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (START_DIR, *START_DIR.parents) if (path / 'pyproject.toml').exists())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from analytics.attacks.mechanistic_propagation import (
    DEFAULT_ATTACK_DIR, DEFAULT_MODEL_ROOT, DEFAULT_OUTPUT_DIR,
    build_stage_effect_table, common_correct_audit_positions,
    crawl_model_dirs, default_model_specs, find_norm_harm_counterexamples,
    load_attack_aggregates, plot_attack_outcomes, plot_block_diagrams,
    plot_layerwise_gain, plot_margin_trajectories, plot_source_target_asr,
    summarize_attack_metrics, summarize_traces, trace_models,
    write_analysis_tables,
)

MODEL_ROOT = PROJECT_ROOT / DEFAULT_MODEL_ROOT
ATTACK_DIR = PROJECT_ROOT / DEFAULT_ATTACK_DIR
OUTPUT_DIR = PROJECT_ROOT / DEFAULT_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 0
EPSILONS = (0.05, 0.10, 0.30)
SELECTED_KEYS = (
    'cnn', 'resnet18', 'mlp', 'mlpmixer',
    'hvql', 'hpvql', 'hvqm', 'hqe', 'hquanv',
    'drophvql', 'reshvql', 'vqm50', 'vqm100', 'vqm200',
)
OUTPUT_DIR

WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation')

## 2. Crawl and find models/artifacts

Discovery and selection remain separate. `.model` anchors define valid trained-model folders; the attack chunks define the evaluated source-target population.

In [2]:
all_model_dirs = crawl_model_dirs(MODEL_ROOT)
all_specs = default_model_specs()
attack_table = load_attack_aggregates(ATTACK_DIR)

print(f'Discovered {len(all_model_dirs)} .model-anchored folders under {MODEL_ROOT}')
print(f'Attack table: {len(attack_table):,} rows, {attack_table.source_model.nunique()} sources, {attack_table.target_model.nunique()} targets, seeds={sorted(attack_table.seed.unique())}')
print('Selected candidates (not loaded yet):')
pd.DataFrame([
    {'key': key, 'display': all_specs[key].display_name, 'attack_name': all_specs[key].attack_name,
     'model_dir': all_specs[key].model_dir, 'artifact_note': all_specs[key].artifact_note}
    for key in SELECTED_KEYS
])

Attack tables:   0%|          | 0/74 [00:00<?, ?it/s]

Discovered 51 .model-anchored folders under C:\Users\fwilh\Desktop\QB-PIER\logs\ByModel
Attack table: 57,498 rows, 37 sources, 37 targets, seeds=[np.int64(0), np.int64(42)]
Selected candidates (not loaded yet):


,key,display,attack_name,model_dir,artifact_note
0,cnn,CNN,"Classical/convnet/conv_channels=[32,64],fc_siz...",logs/ByModel/Classical/convnet/conv_channels=[...,
1,resnet18,ResNet18,Classical/resnet18,logs/ByModel/Classical/resnet18,
2,mlp,MLP (128-64),"Classical/mlp/hidden_sizes=[128,64]","logs/ByModel/Classical/mlp/hidden_sizes=[128,64]",The higher-capacity folder named hidden_sizes=...
3,mlpmixer,MLPMixer,Classical/mlpmixer,logs/ByModel/Classical/mlpmixer,
4,hvql,"HVQL (raw artifact: HVQM, Strong d=50)","Hybrid/hvqm/ansatz=strong,qdepth=50",logs/ByBenchmark/HVQM-Benchmarks/ansatz=strong...,"Historical attack path says HVQM, but the chec..."
5,hpvql,"HPVQL (5 blocks, d=50)",Hybrid/hpvql,logs/ByBenchmark/Baseline-Models/Hybrid/hpvql,The available checkpoint matches all audit pre...
6,hvqm,"HVQM (raw artifact: HVQL, Strong d=50)","Hybrid/hvql/ansatz=strong,qdepth=50","logs/ByBenchmark/HVQL-Ansatzs/ansatz=strong,qd...","Historical attack path says HVQL, but the chec..."
7,hqe,HQE,Hybrid/hqe,logs/ByBenchmark/Baseline-Models/Hybrid/hqe,
8,hquanv,HQuanv,Hybrid/hquanv,logs/ByBenchmark/Baseline-Models/Hybrid/hquanv,
9,drophvql,Dropout HVQL,Hybrid/hvql/dropout,logs/ByModel/Hybrid/hvql/dropout,


## 3. Load models or prepare data

This is the expensive block. It loads only the manually selected variants, validates each checkpoint against saved clean logits, and traces self, representative transferred, and L2-matched random perturbations.

In [3]:
selected_specs = [all_specs[key] for key in SELECTED_KEYS]
common_positions, common_dataset_indices, common_labels = common_correct_audit_positions(ATTACK_DIR, selected_specs, seed=SEED)
print(f'Common clean-correct audit cohort: n={len(common_positions)} / 30; labels={common_labels.tolist()}')

traces, checkpoint_alignment, architecture_map, direction_similarity, effective_rank = trace_models(
    PROJECT_ROOT, ATTACK_DIR, selected_specs, seed=SEED, epsilons=EPSILONS
)
trace_summary = summarize_traces(traces)
variant_seed_metrics, architecture_metrics = summarize_attack_metrics(attack_table)
stage_effects = build_stage_effect_table(trace_summary, epsilon=0.10)
counterexamples = find_norm_harm_counterexamples(trace_summary, epsilon=0.10)

table_paths = write_analysis_tables(
    OUTPUT_DIR, attack_table=attack_table, variant_seed=variant_seed_metrics,
    architecture_summary=architecture_metrics, traces=traces, trace_summary=trace_summary,
    alignments=checkpoint_alignment, architecture_map=architecture_map,
    direction_similarity=direction_similarity, effective_rank=effective_rank,
    stage_effects=stage_effects, counterexamples=counterexamples,
)
print(checkpoint_alignment[['model', 'alignment', 'mean_abs_logit_error', 'prediction_match']].to_string(index=False))
print(f'Trace rows: {len(traces):,}; summary rows: {len(trace_summary):,}; counterexamples: {len(counterexamples):,}')
table_paths

Common clean-correct audit cohort: n=19 / 30; labels=[7, 2, 1, 0, 4, 1, 9, 0, 9, 0, 1, 7, 4, 6, 7, 3, 3, 2, 8]


Target models:   0%|          | 0/14 [00:00<?, ?it/s]

Error loading metrics: Metrics file not found at C:\Users\fwilh\Desktop\QB-PIER\logs\ByBenchmark\Baseline-Models\Quantum\VQM200\0\metrics\logger.pkl
Traceback (most recent call last):
  File "C:\Users\fwilh\Desktop\QB-PIER\src\metrics\MetricsContainer.py", line 182, in load_metrics
    raise FileNotFoundError(f"Metrics file not found at {pkl_path}")
FileNotFoundError: Metrics file not found at C:\Users\fwilh\Desktop\QB-PIER\logs\ByBenchmark\Baseline-Models\Quantum\VQM200\0\metrics\logger.pkl


                                 model  alignment  mean_abs_logit_error  prediction_match
                                   CNN      exact              0.001094               1.0
                              ResNet18      exact              0.000512               1.0
                          MLP (128-64)      exact              0.001092               1.0
                              MLPMixer      exact              0.000355               1.0
HVQL (raw artifact: HVQM, Strong d=50)      exact              0.001119               1.0
                HPVQL (5 blocks, d=50) near-match              0.105618               1.0
HVQM (raw artifact: HVQL, Strong d=50)      exact              0.000464               1.0
                                   HQE      exact              0.001001               1.0
                                HQuanv      exact              0.002826               1.0
                          Dropout HVQL      exact              0.001205               1.0
          

{'attack_aggregates': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/attack_aggregates.csv'),
 'variant_seed_metrics': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/variant_seed_metrics.csv'),
 'architecture_metrics': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/architecture_metrics.csv'),
 'layerwise_trace_samples': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/layerwise_trace_samples.csv'),
 'layerwise_trace_summary': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/layerwise_trace_summary.csv'),
 'checkpoint_alignment': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/checkpoint_alignment.csv'),
 'architecture_map': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/architecture_map.csv'),
 'source_direction_similarity': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_pro

## 4. Generate the figure/table

Figures use dimension-normalized RMS displacement gain for cross-layer comparisons. Margin trajectories are downstream-linearized at intermediate blocks and exact only at final logits.

In [4]:
figure_paths = {}
fig, paths = plot_attack_outcomes(variant_seed_metrics, selected_specs, OUTPUT_DIR / 'fig01_attack_outcomes')
figure_paths['attack_outcomes'] = paths
plt.close(fig)
fig, paths = plot_block_diagrams(architecture_map, OUTPUT_DIR / 'fig02_block_diagrams')
figure_paths['block_diagrams'] = paths
plt.close(fig)
for name, keys in {
    'fig03a_classical_gain': ('cnn', 'resnet18', 'mlp', 'mlpmixer'),
    'fig03b_late_hybrid_gain': ('hvql', 'hpvql', 'hvqm', 'hqe'),
    'fig03c_quantum_local_gain': ('hquanv', 'vqm50', 'vqm100', 'vqm200'),
    'fig03d_hvql_variants_gain': ('hvql', 'drophvql', 'reshvql'),
}.items():
    fig, paths = plot_layerwise_gain(trace_summary, keys, OUTPUT_DIR / name, epsilon=0.10)
    figure_paths[name] = paths
    plt.close(fig)
fig, paths = plot_margin_trajectories(
    trace_summary, ('hvql', 'hpvql', 'hvqm', 'hqe'), OUTPUT_DIR / 'fig04_margin_trajectories', epsilon=0.10
)
figure_paths['margin_trajectories'] = paths
plt.close(fig)
fig, paths, asr_matrix = plot_source_target_asr(
    attack_table, selected_specs, OUTPUT_DIR / 'fig05_source_target_asr', epsilon=0.10
)
figure_paths['source_target_asr'] = paths
plt.close(fig)
asr_matrix.to_csv(OUTPUT_DIR / 'selected_source_target_asr_e010.csv')

canonical = architecture_metrics[architecture_metrics.architecture.isin(['HVQL', 'HPVQL', 'HVQM', 'HQE', 'HQuanv'])]
display_cols = [
    'architecture', 'robust_accuracy_000__mean', 'robust_accuracy_010__mean',
    'retention_010__mean', 'robust_accuracy_auc__mean', 'whitebox_asr_010__mean',
    'incoming_asr_010__mean', 'outgoing_asr_010__mean',
]
print(canonical[display_cols].sort_values('robust_accuracy_010__mean', ascending=False).to_string(index=False))
print('\nStage effects:')
print(stage_effects.to_string(index=False))
print('\nStrongest norm/visibility counterexamples:')
print(counterexamples.head(20).to_string(index=False))
figure_paths

architecture  robust_accuracy_000__mean  robust_accuracy_010__mean  retention_010__mean  robust_accuracy_auc__mean  whitebox_asr_010__mean  incoming_asr_010__mean  outgoing_asr_010__mean
        HVQL                   0.991720                    0.73982             0.745995                   0.244681                0.254005                0.055002                0.128727
       HPVQL                   0.992300                    0.71045             0.715991                   0.239979                0.284009                0.054851                0.147120
         HQE                   0.983900                    0.62125             0.631440                   0.509421                0.368560                0.093602                0.145600
        HVQM                   0.870483                    0.46070             0.521148                   0.153265                0.478982                0.119037                0.106380
      HQuanv                   0.975750                    0.2992

{'attack_outcomes': {'png': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig01_attack_outcomes.png'),
  'pdf': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig01_attack_outcomes.pdf')},
 'block_diagrams': {'png': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig02_block_diagrams.png'),
  'pdf': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig02_block_diagrams.pdf')},
 'fig03a_classical_gain': {'png': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig03a_classical_gain.png'),
  'pdf': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig03a_classical_gain.pdf')},
 'fig03b_late_hybrid_gain': {'png': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig03b_late_hybrid_gain.png'),
  'pdf': WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/outputs/mechanistic_propagation/fig03b_late_hybrid_gain.p